# FOLIO Lean Improved: Error Classification Analysis

Comprehensive analysis of 7 verified-but-wrong cases using error root cause classification.

**Improvement Results**: With stricter anti-axiomatization prompts:
- Axiomatization errors dropped from **77.6%** (38/49) to **14.3%** (1/7)
- Verification rate dropped from **96%** to **20.20%** (trade-off: harder to generate valid Lean)
- Total verified-but-wrong cases reduced from **49** to **7**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Define consistent color palette
ERROR_COLORS = {
    'AXIOMATIZES_CONCLUSION': '#e74c3c',
    'AXIOMATIZES_CONTRADICTION': '#3498db',
    'INCORRECT_FORMALIZATION': '#f39c12',
    'REASONING_FAILURE': '#2ecc71',
    'AXIOMATIZES_UNMENTIONED': '#9b59b6',
    'OTHER': '#95a5a6'
}

## 1. Load Data and Basic Statistics

In [ ]:
# Load the error analysis CSV
df = pd.read_csv('../../results/folio/lean_improved/error_root_cause_analysis.csv')

print(f"Total verified-but-wrong cases: {len(df)}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData shape: {df.shape}")
print(f"\nUnique error types: {df['Root_Cause_Category'].nunique()}")
print(f"\nUnique patterns: {df['Pattern'].nunique()}")

In [ ]:
# Show first few rows
df.head()

## 2. Error Type Distribution

In [ ]:
# Calculate error distribution
error_dist = df['Root_Cause_Category'].value_counts()

print("Root Cause Distribution:")
print("=" * 70)
for category, count in error_dist.items():
    percentage = (count / len(df)) * 100
    print(f"{category:30s} {count:3d} cases ({percentage:5.1f}%)")
print(f"\nTotal: {len(df)} cases")

# Calculate axiomatization total
axiom_types = ['AXIOMATIZES_CONCLUSION', 'AXIOMATIZES_CONTRADICTION', 'AXIOMATIZES_UNMENTIONED']
axiom_total = sum(error_dist.get(t, 0) for t in axiom_types)
print(f"\nTotal Axiomatization Errors: {axiom_total} ({axiom_total/len(df)*100:.1f}%)")

In [ ]:
# Visualize error distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = [ERROR_COLORS.get(cat, '#95a5a6') for cat in error_dist.index]
error_dist.plot(kind='bar', ax=ax1, color=colors, edgecolor='black', linewidth=1.2)
ax1.set_title('Error Type Distribution (Bar Chart) - Improved Prompts', fontsize=14, fontweight='bold')
ax1.set_xlabel('Error Category', fontsize=12)
ax1.set_ylabel('Number of Cases', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Add count labels on bars
for i, v in enumerate(error_dist.values):
    ax1.text(i, v + 0.1, str(v), ha='center', va='bottom', fontweight='bold')

# Pie chart
colors_pie = [ERROR_COLORS.get(cat, '#95a5a6') for cat in error_dist.index]
ax2.pie(error_dist.values, labels=error_dist.index, autopct='%1.1f%%', 
        colors=colors_pie, startangle=90, textprops={'fontsize': 10})
ax2.set_title('Error Type Distribution (Pie Chart) - Improved Prompts', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Comparison with Original Prompts

In [ ]:
# Load original results for comparison
df_original = pd.read_csv('../../results/folio/lean/error_root_cause_analysis.csv')

print("COMPARISON: Original vs Improved Prompts")
print("=" * 70)
print(f"\nTotal verified-but-wrong cases:")
print(f"  Original:  {len(df_original)} cases")
print(f"  Improved:  {len(df)} cases")
print(f"  Reduction: {len(df_original) - len(df)} cases ({(len(df_original) - len(df))/len(df_original)*100:.1f}%)")

# Compare axiomatization rates
orig_axiom_types = ['AXIOMATIZES_CONCLUSION', 'AXIOMATIZES_CONTRADICTION', 'AXIOMATIZES_UNMENTIONED']
orig_error_dist = df_original['Root_Cause_Category'].value_counts()
orig_axiom_total = sum(orig_error_dist.get(t, 0) for t in orig_axiom_types)

print(f"\nAxiomatization errors:")
print(f"  Original:  {orig_axiom_total}/{len(df_original)} ({orig_axiom_total/len(df_original)*100:.1f}%)")
print(f"  Improved:  {axiom_total}/{len(df)} ({axiom_total/len(df)*100:.1f}%)")
print(f"  Reduction: {orig_axiom_total/len(df_original)*100 - axiom_total/len(df)*100:.1f} percentage points")

print("\nKey insight: Stricter anti-axiomatization rules dramatically reduced")
print("axiomatization errors from 77.6% to 14.3%!")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))

# Prepare data for comparison
comparison_data = pd.DataFrame({
    'Original': [orig_axiom_total/len(df_original)*100, 
                 100 - orig_axiom_total/len(df_original)*100],
    'Improved': [axiom_total/len(df)*100, 
                 100 - axiom_total/len(df)*100]
}, index=['Axiomatization Errors', 'Other Errors'])

comparison_data.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'], 
                     edgecolor='black', linewidth=1.2, width=0.7)
ax.set_title('Axiomatization Error Rate: Original vs Improved Prompts', 
             fontsize=14, fontweight='bold')
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_xlabel('Error Category', fontsize=12)
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
ax.legend(title='Prompt Version', loc='upper right')

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.tight_layout()
plt.show()

## 4. Error Types by Prediction Pattern

In [ ]:
# Cross-tabulation of patterns and error types
pattern_error = pd.crosstab(df['Pattern'], df['Root_Cause_Category'])

print("Error Types by Pattern:")
print("=" * 70)
print(pattern_error)

print("\nKey insights:")
for pattern in pattern_error.index:
    top_error = pattern_error.loc[pattern].idxmax()
    count = pattern_error.loc[pattern].max()
    total = pattern_error.loc[pattern].sum()
    print(f"  {pattern}: Most common is {top_error} ({count}/{total} cases)")

## 5. Key Findings

In [ ]:
print("=" * 70)
print("KEY FINDINGS: FOLIO LEAN IMPROVED ERROR CLASSIFICATION")
print("=" * 70)

# Top error types
print("\n1. ERROR TYPE DISTRIBUTION:")
for i, (cat, count) in enumerate(error_dist.items(), 1):
    pct = count / len(df) * 100
    print(f"   #{i}: {cat}")
    print(f"       {count} cases ({pct:.1f}%)")

print(f"\n2. DRAMATIC IMPROVEMENT IN AXIOMATIZATION:")
print(f"   Original axiomatization rate: 77.6% (38/49 cases)")
print(f"   Improved axiomatization rate: {axiom_total/len(df)*100:.1f}% ({axiom_total}/{len(df)} cases)")
print(f"   Reduction: {orig_axiom_total/len(df_original)*100 - axiom_total/len(df)*100:.1f} percentage points")

print(f"\n3. TRADE-OFF: VERIFICATION RATE")
print(f"   Stricter prompts make it harder to generate valid Lean code:")
print(f"   - Original verification rate: 96% (195/203)")
print(f"   - Improved verification rate: 20.20% (41/203)")
print(f"   - But verified-but-wrong cases dropped: 49 → 7")

print(f"\n4. NEW ERROR DISTRIBUTION:")
print(f"   With axiomatization suppressed, we now see:")
print(f"   - INCORRECT_FORMALIZATION: {error_dist.get('INCORRECT_FORMALIZATION', 0)} cases (premise translation errors)")
print(f"   - AXIOMATIZES_CONTRADICTION: {error_dist.get('AXIOMATIZES_CONTRADICTION', 0)} cases (still some cheating)")
print(f"   - AXIOMATIZES_CONCLUSION: {error_dist.get('AXIOMATIZES_CONCLUSION', 0)} case (rare now!)")

print("\n" + "=" * 70)
print("CONCLUSION:")
print("=" * 70)
print("\nThe improved prompts successfully addressed the axiomatization problem,")
print("reducing it from 77.6% to 14.3%. However, this came at the cost of")
print("making Lean code generation much harder (96% → 20% verification rate).")
print("The remaining errors are mostly genuine formalization issues rather than")
print("attempts to cheat by axiomatizing conclusions.")
print("=" * 70)

## 6. Example Cases by Error Type

### 6.1 INCORRECT_FORMALIZATION (Most Common)

In [ ]:
# Get INCORRECT_FORMALIZATION examples
incorrect_form = df[df['Root_Cause_Category'] == 'INCORRECT_FORMALIZATION']

print(f"Total INCORRECT_FORMALIZATION cases: {len(incorrect_form)}")
print("\nExample cases:")
print("=" * 70)

for idx, (_, row) in enumerate(incorrect_form.iterrows(), 1):
    print(f"\nExample {idx}: Case #{row['Example_ID']}")
    print(f"  Pattern: {row['Pattern']}")
    print(f"  Premises: {row['Premises'][:150]}...")
    print(f"  Conclusion: {row['Conclusion'][:80]}...")
    print(f"  Problematic Line: {row['Problematic_Lines']}")
    print(f"  Error: {row['Error_Description'][:150]}...")

### 6.2 AXIOMATIZES_CONTRADICTION

In [ ]:
# Get AXIOMATIZES_CONTRADICTION examples
axiom_contradiction = df[df['Root_Cause_Category'] == 'AXIOMATIZES_CONTRADICTION']

print(f"Total AXIOMATIZES_CONTRADICTION cases: {len(axiom_contradiction)}")
print("\nExample cases:")
print("=" * 70)

for idx, (_, row) in enumerate(axiom_contradiction.iterrows(), 1):
    print(f"\nExample {idx}: Case #{row['Example_ID']}")
    print(f"  Pattern: {row['Pattern']}")
    print(f"  Premises: {row['Premises'][:150]}...")
    print(f"  Conclusion: {row['Conclusion'][:80]}...")
    print(f"  Problematic Line: {row['Problematic_Lines']}")
    print(f"  Error: {row['Error_Description'][:150]}...")
    print(f"  Axiom: {row['Specific_Axiom'][:100]}...")

### 6.3 AXIOMATIZES_CONCLUSION (Now Rare!)

In [ ]:
# Get AXIOMATIZES_CONCLUSION examples
axiom_conclusion = df[df['Root_Cause_Category'] == 'AXIOMATIZES_CONCLUSION']

print(f"Total AXIOMATIZES_CONCLUSION cases: {len(axiom_conclusion)}")
print("\nExample cases:")
print("=" * 70)

for idx, (_, row) in enumerate(axiom_conclusion.iterrows(), 1):
    print(f"\nExample {idx}: Case #{row['Example_ID']}")
    print(f"  Pattern: {row['Pattern']}")
    print(f"  Premises: {row['Premises'][:150]}...")
    print(f"  Conclusion: {row['Conclusion'][:80]}...")
    print(f"  Problematic Line: {row['Problematic_Lines']}")
    print(f"  Error: {row['Error_Description'][:150]}...")
    print(f"  Axiom: {row['Specific_Axiom'][:100]}...")

## 7. Summary Statistics

In [ ]:
print("=" * 70)
print("SUMMARY: IMPROVED PROMPTS EFFECTIVENESS")
print("=" * 70)

print("\n✓ SUCCESS METRICS:")
print(f"  - Axiomatization errors: 77.6% → {axiom_total/len(df)*100:.1f}% (63.3pp reduction!)")
print(f"  - Verified-but-wrong cases: 49 → {len(df)} (85.7% reduction)")
print(f"  - Most errors now genuine formalization issues, not cheating")

print("\n⚠️  TRADE-OFF:")
print(f"  - Verification rate: 96% → 20.20% (75.8pp drop)")
print(f"  - Stricter rules make valid Lean code harder to generate")
print(f"  - Many syntactically correct proofs now rejected")

print("\n📊 ERROR BREAKDOWN (7 cases):")
for cat, count in error_dist.items():
    pct = count / len(df) * 100
    print(f"  - {cat}: {count} ({pct:.1f}%)")

print("\n" + "=" * 70)